# Irrigation Need — Classification Pipeline
**Playground Series S6E4 | Multiclass: Low / Medium / High**

Models: DecisionTree · GaussianNB · LogisticRegression · KMeans (as classifier) · RandomForest  
Evaluation: StratifiedKFold (5 folds) · Accuracy · Macro F1

## Cell 1 — Colab / Local Path Setup

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Update this path to match your Google Drive folder location
    BASE = '/content/drive/MyDrive/irrigation-kaggle-assignment'
else:
    BASE = '.'

DATA_DIR   = f'{BASE}/data'
OUTPUT_DIR = f'{BASE}/outputs'

print(f'Running in: {"Colab" if IN_COLAB else "Local"}')
print(f'Data dir:   {DATA_DIR}')

## Cell 2 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

SEED = 42
print('Imports OK')

## Cell 3 — Create Output Directories

In [ ]:
# Create all output folders upfront so later save calls never fail
for folder in ['metrics', 'figures', 'submissions']:
    path = os.path.join(OUTPUT_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(f'Ready: {path}')

## Cell 4 — Load Data

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train : {train.shape}')
print(f'Test  : {test.shape}')
print(f'Sub   : {sub.shape}')
train.head()

## Cell 5 — Data Validation

In [ ]:
# Quick sanity checks before any processing
print('=== Missing values ===')
print('Train:', train.isnull().sum().sum())
print('Test: ', test.isnull().sum().sum())

print('\n=== Target distribution ===')
print(train['Irrigation_Need'].value_counts())

print('\n=== Column dtypes ===')
print(train.dtypes)

## Cell 6 — Encode Categorical Features

In [ ]:
TARGET = 'Irrigation_Need'

# Identify categorical columns (object dtype), excluding the target
cat_cols = [
    c for c in train.select_dtypes(include='object').columns
    if c != TARGET
]

# Fit LabelEncoder on train+test combined to avoid unseen-label errors at inference
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined_vals = pd.concat([train[col], test[col]], axis=0)
    le.fit(combined_vals)
    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])
    encoders[col] = le

print(f'Encoded {len(cat_cols)} columns: {cat_cols}')

## Cell 7 — Encode Target Labels

In [ ]:
# Fixed mapping preserves the ordinal meaning: Low=0, Medium=1, High=2
label_map = {'Low': 0, 'Medium': 1, 'High': 2}
inv_map   = {v: k for k, v in label_map.items()}

y = train[TARGET].map(label_map).values

print('Label map:', label_map)
print('Class counts:', dict(zip(*np.unique(y, return_counts=True))))

## Cell 8 — Feature Split and Scaling

In [ ]:
# Drop id (row identifier) and target from features
feature_cols = [c for c in train.columns if c not in ['id', TARGET]]

X_raw      = train[feature_cols].values
X_test_raw = test[feature_cols].values

# StandardScaler is required for LogisticRegression and GaussianNB to work correctly,
# and is necessary for KMeans (distance-based). Tree models are scale-invariant
# but scaling does not hurt them — one scaler for all models keeps the pipeline uniform.
scaler = StandardScaler()
X           = scaler.fit_transform(X_raw)
X_test      = scaler.transform(X_test_raw)

print(f'Feature matrix : {X.shape}')
print(f'Test matrix    : {X_test.shape}')
print(f'Features used  : {feature_cols}')

## Cell 9 — KMeans Classifier

In [ ]:
# KMeans is unsupervised, so we adapt it into a classifier with this wrapper:
#   1. Fit KMeans with n_clusters=3 (matching our 3 classes)
#   2. For each cluster, find the majority label among training samples in that cluster
#   3. At predict time: assign sample to nearest cluster → return that cluster's majority label
#
# This is a reasonable heuristic but KMeans has no guarantee that clusters align with classes.

class KMeansClassifier(BaseEstimator, ClassifierMixin):
    """Wraps KMeans as a classifier via cluster-to-majority-label mapping."""

    def __init__(self, n_clusters=3, random_state=42):
        self.n_clusters   = n_clusters
        self.random_state = random_state

    def fit(self, X, y):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            random_state=self.random_state,
            n_init=10
        )
        self.kmeans_.fit(X)

        cluster_assignments = self.kmeans_.labels_
        self.cluster_map_ = {}

        for cluster_id in range(self.n_clusters):
            mask = cluster_assignments == cluster_id
            if mask.sum() == 0:
                # Empty cluster: default to class 0
                self.cluster_map_[cluster_id] = 0
            else:
                counts = np.bincount(y[mask], minlength=len(np.unique(y)))
                self.cluster_map_[cluster_id] = int(np.argmax(counts))

        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        cluster_ids = self.kmeans_.predict(X)
        return np.array([self.cluster_map_[c] for c in cluster_ids])


print('KMeansClassifier defined.')

## Cell 10 — Define All Models

In [ ]:
# class_weight='balanced' compensates for the skewed distribution
# (High is only 3.3% of data — without balancing, models ignore it).
# KMeans and GaussianNB do not support class_weight natively.

models = {
    'DecisionTree': DecisionTreeClassifier(
        random_state=SEED,
        class_weight='balanced'
    ),
    'GaussianNB': GaussianNB(),
    'LogisticReg': LogisticRegression(
        max_iter=1000,          # ensures convergence on large dataset
        random_state=SEED,
        class_weight='balanced',
        C=1.0
    ),
    'KMeans': KMeansClassifier(
        n_clusters=3,
        random_state=SEED
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        class_weight='balanced',
        n_jobs=-1               # use all CPU cores
    ),
}

print('Models registered:')
for name in models:
    print(f'  {name}')

## Cell 11 — Cross-Validation

In [ ]:
# StratifiedKFold keeps class proportions intact in each fold —
# critical here because High (3.3%) would vanish in some folds without stratification.
#
# We track both accuracy and macro F1:
#   - Accuracy: easy to interpret
#   - Macro F1: actual competition metric (all classes weighted equally)

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []

for name, model in models.items():
    print(f'Evaluating: {name} ...', end=' ')

    fold_acc = []
    fold_f1  = []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)

        fold_acc.append(accuracy_score(y_val, y_pred))
        fold_f1.append(f1_score(y_val, y_pred, average='macro'))

    mean_acc = np.mean(fold_acc)
    mean_f1  = np.mean(fold_f1)

    results.append({
        'Model':       name,
        'CV_Accuracy': round(mean_acc, 4),
        'CV_Acc_Std':  round(np.std(fold_acc), 4),
        'CV_MacroF1':  round(mean_f1, 4),
        'CV_F1_Std':   round(np.std(fold_f1), 4),
    })

    print(f'Acc={mean_acc:.4f}  MacroF1={mean_f1:.4f}')

print('\nCross-validation complete.')

## Cell 12 — Save CV Results

In [ ]:
results_df = pd.DataFrame(results).sort_values('CV_MacroF1', ascending=False).reset_index(drop=True)

save_path = os.path.join(OUTPUT_DIR, 'metrics', 'results.csv')
results_df.to_csv(save_path, index=False)
print(f'Saved: {save_path}')

results_df

## Cell 13 — Confusion Matrices

In [ ]:
# Use a single stratified 80/20 split for visualisation only.
# CV scores are the reliable numbers; this is just for the plots.

CLASS_NAMES = ['Low', 'Medium', 'High']

X_tr80, X_val20, y_tr80, y_val20 = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

for name, model in models.items():
    model.fit(X_tr80, y_tr80)
    y_pred = model.predict(X_val20)

    cm   = confusion_matrix(y_val20, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nMacro F1 = {f1_score(y_val20, y_pred, average="macro"):.4f}')
    plt.tight_layout()

    fig_path = os.path.join(OUTPUT_DIR, 'figures', f'cm_{name}.png')
    plt.savefig(fig_path, dpi=150)
    plt.close()
    print(f'Saved: {fig_path}')

## Cell 14 — Select Best Model and Retrain on Full Data

In [ ]:
# Best model = highest CV macro F1 (the competition metric)
best_row        = results_df.iloc[0]
best_model_name = best_row['Model']
best_model      = models[best_model_name]

print(f'Best model : {best_model_name}')
print(f'CV Macro F1: {best_row["CV_MacroF1"]}  (±{best_row["CV_F1_Std"]})')
print(f'CV Accuracy: {best_row["CV_Accuracy"]}  (±{best_row["CV_Acc_Std"]})')

# Retrain on the full training set — more data reduces variance and usually improves score
best_model.fit(X, y)
print(f'\nRetrained {best_model_name} on all {X.shape[0]:,} training samples.')

## Cell 15 — Generate Submission

In [ ]:
y_test_pred   = best_model.predict(X_test)
y_test_labels = [inv_map[p] for p in y_test_pred]

sub['Irrigation_Need'] = y_test_labels

sub_path = os.path.join(OUTPUT_DIR, 'submissions', 'submission.csv')
sub.to_csv(sub_path, index=False)

print(f'Submission saved: {sub_path}')
print('\nPrediction distribution:')
print(sub['Irrigation_Need'].value_counts())
sub.head(10)

---
## Notes

### Model Selection Logic
The best model is chosen by **CV macro F1** — the same metric Kaggle uses. Macro F1 weights all three classes equally, so a model that handles the rare `High` class well is favoured over one that simply predicts `Low` all the time. RandomForest is expected to win because it reduces variance through bagging and handles mixed feature types well, while `class_weight='balanced'` forces it to pay attention to the minority `High` class.

### One Key Limitation
The KMeans classifier is fundamentally misaligned with this problem. KMeans clusters by geometric proximity in feature space — there is no reason those clusters correspond to irrigation need levels. The cluster-to-label mapping is a post-hoc heuristic and will likely perform close to random on the `High` class. It is included here to demonstrate the concept, not because it is competitive.